# 🏆 Mini Project: Scouting the Academy's Next Star (NumPy)

### The Scenario

You are a **data analyst for a sports academy**. The head coach is putting together a **6-player squad** for an upcoming regional tournament, and has given you a fixed set of rules to follow so every analyst on the team produces a consistent, comparable report.

You have performance data for 12 players across 4 physical metrics. Follow the steps below exactly — the weights and selection rule are fixed by the coaching staff, so your final squad should match a single correct answer. Your own input comes in at the very end, in how you *explain* the result.

**Deliverable:** a ranked, structured record of players, the final 6-player squad (following the fixed rule below), and a short written justification.


## 1. The Data

Run the cell below — this is your starting dataset. Don't change it directly; if you need to adjust anything, work on a copy.

| Column | Meaning |
|---|---|
| 0 | Sprint Speed (km/h) |
| 1 | Endurance Score (0–100) |
| 2 | Strength Score (0–100) |
| 3 | Agility Score (0–100) |


In [87]:
import numpy as np

players = np.array([
    "Adam", "Mostafa", "Yara", "Hana", "Ziad", "Malak",
    "Omar", "Farida", "Karim", "Nada", "Tarek", "Reem"
])

positions = np.array([
    "Forward", "Defender", "Midfielder", "Forward", "Goalkeeper", "Defender",
    "Midfielder", "Forward", "Defender", "Midfielder", "Forward", "Goalkeeper"
])

# 12 players x 4 metrics: [Sprint Speed, Endurance, Strength, Agility]
performance = np.array([
    [28.5, 82, 75, 88],
    [26.0, 90, 88, 70],
    [30.2, 78, 60, 92],
    [27.8, 85, 70, 80],
    [24.5, 95, 92, 65],
    [25.9, 70, 65, 75],
    [29.1, 80, 55, 90],
    [31.0, 60, 50, 95],
    [23.8, 88, 90, 60],
    [28.9, 83, 68, 85],
    [27.2, 76, 72, 78],
    [22.9, 93, 95, 58],
])

print(players.shape, positions.shape, performance.shape)

(12,) (12,) (12, 4)


## 2. Explore & Clean

Before ranking anyone, get familiar with the data — and fix a known issue.

1. Look at the shape and structure of `performance`. Pick a couple of players and compare their rows by eye using indexing.
2. There is a known data-entry glitch: **any Strength score above 90 should be capped at exactly 90.** Fix this on a **copy** of `performance` (call it `performance_clean`), so you don't lose the original data.

   💡 `np.clip(array, min, max)` caps values into a range — pass `None` for a bound you don't want to limit.


In [88]:
# Explore the data, then create performance_clean here
print(performance.size)
print(performance.shape)
print(performance.ndim)
print(performance.dtype)
print(performance.itemsize)

performance_clean= performance.copy()
print(performance_clean)



print(performance_clean[:,2].min())

performance_clean[:,2] = np.clip(performance_clean[:,2], None, 90)
print(performance_clean)



48
(12, 4)
2
float64
8
[[28.5 82.  75.  88. ]
 [26.  90.  88.  70. ]
 [30.2 78.  60.  92. ]
 [27.8 85.  70.  80. ]
 [24.5 95.  92.  65. ]
 [25.9 70.  65.  75. ]
 [29.1 80.  55.  90. ]
 [31.  60.  50.  95. ]
 [23.8 88.  90.  60. ]
 [28.9 83.  68.  85. ]
 [27.2 76.  72.  78. ]
 [22.9 93.  95.  58. ]]
50.0
[[28.5 82.  75.  88. ]
 [26.  90.  88.  70. ]
 [30.2 78.  60.  92. ]
 [27.8 85.  70.  80. ]
 [24.5 95.  90.  65. ]
 [25.9 70.  65.  75. ]
 [29.1 80.  55.  90. ]
 [31.  60.  50.  95. ]
 [23.8 88.  90.  60. ]
 [28.9 83.  68.  85. ]
 [27.2 76.  72.  78. ]
 [22.9 93.  90.  58. ]]


## 3. Build the Talent Score

The coaching staff has already agreed on how much each metric should count toward a player's overall Talent Score, so every scout's numbers line up. This tournament is a fast-paced format, so Sprint Speed and Agility are weighted highest:

| Metric | Weight |
|---|---|
| Sprint Speed | ×1.5 |
| Endurance | ×0.7 |
| Strength | ×0.5 |
| Agility | ×1.3 |

Combine `performance_clean` with this weight array using **broadcasting**, then sum across the metrics (per player) to get one `talent_score` value per player.


In [89]:
weights = np.array([1.5, 0.7, 0.5, 1.3])  # Sprint Speed, Endurance, Strength, Agility

# Compute talent_score here (should be a 1D array of 12 values)
performance_clean=performance_clean*weights
print(performance_clean)

talent_score=performance_clean.sum(axis=1)

print(talent_score)


[[ 42.75  57.4   37.5  114.4 ]
 [ 39.    63.    44.    91.  ]
 [ 45.3   54.6   30.   119.6 ]
 [ 41.7   59.5   35.   104.  ]
 [ 36.75  66.5   45.    84.5 ]
 [ 38.85  49.    32.5   97.5 ]
 [ 43.65  56.    27.5  117.  ]
 [ 46.5   42.    25.   123.5 ]
 [ 35.7   61.6   45.    78.  ]
 [ 43.35  58.1   34.   110.5 ]
 [ 40.8   53.2   36.   101.4 ]
 [ 34.35  65.1   45.    75.4 ]]
[252.05 237.   249.5  240.2  232.75 217.85 244.15 237.   220.3  245.95
 231.4  219.85]


## 4. Package Everything into a Structured Array

Right now you have three separate arrays (`players`, `positions`, `talent_score`) that all describe the same 12 people — that's exactly the situation structured arrays are built for.

1. Build a structured array called `Squad` combining all three, with fields: `name` (string), `position` (string), and `score` (float).

   💡 One way: build it from a list of tuples using `dtype=[("name","U10"), ("position","U12"), ("score","f4")]`. You can construct the list of tuples with a loop over the index, e.g. `[(players[i], positions[i], talent_score[i]) for i in range(len(players))]`.

2. Explore it: print the field names, the shape, and confirm you can access `Squad["name"]` and `Squad["score"]` correctly.


In [90]:
# Build the Squad structured array here
squad=np.array([(players[i], positions[i], talent_score[i]) for i in range(len(players))], dtype=[("name","U10"), ("position","U12"), ("score","f4")])

print(squad)

print(squad.dtype.names, squad.shape)

print(squad["name"],squad["score"])


[('Adam', 'Forward', 252.05) ('Mostafa', 'Defender', 237.  )
 ('Yara', 'Midfielder', 249.5 ) ('Hana', 'Forward', 240.2 )
 ('Ziad', 'Goalkeeper', 232.75) ('Malak', 'Defender', 217.85)
 ('Omar', 'Midfielder', 244.15) ('Farida', 'Forward', 237.  )
 ('Karim', 'Defender', 220.3 ) ('Nada', 'Midfielder', 245.95)
 ('Tarek', 'Forward', 231.4 ) ('Reem', 'Goalkeeper', 219.85)]
('name', 'position', 'score') (12,)
['Adam' 'Mostafa' 'Yara' 'Hana' 'Ziad' 'Malak' 'Omar' 'Farida' 'Karim'
 'Nada' 'Tarek' 'Reem'] [252.05 237.   249.5  240.2  232.75 217.85 244.15 237.   220.3  245.95
 231.4  219.85]


## 5. Filter & Sort by Field

Use field-based access to answer these:

1. Which players are Goalkeepers? (filter `Squad` where `position == "Goalkeeper"`)
2. Sort the whole `Squad` array by `score`, from highest to lowest, and print the result. (`np.sort` sorts ascending by default — think about how to reverse it.)
3. Convert `Squad` to a **record array** and use dot notation (`.name`, `.score`) to print the top 3 names and scores from your sorted result.


In [91]:
# Filter, sort, and use a record array here
p=squad['position']
print(squad[p=='Goalkeeper'])

sort_score=np.sort(squad,order='score')[::-1]
print(sort_score)

record_squad = squad.view(np.recarray)
print(record_squad)
print(record_squad.name)
print(record_squad.score)

record_squad2 = sort_score.view(np.recarray)
print(record_squad2)
print(record_squad2[0:3])

print(record_squad2.name[:3])
print(record_squad2.score[:3])

[('Ziad', 'Goalkeeper', 232.75) ('Reem', 'Goalkeeper', 219.85)]
[('Adam', 'Forward', 252.05) ('Yara', 'Midfielder', 249.5 )
 ('Nada', 'Midfielder', 245.95) ('Omar', 'Midfielder', 244.15)
 ('Hana', 'Forward', 240.2 ) ('Mostafa', 'Defender', 237.  )
 ('Farida', 'Forward', 237.  ) ('Ziad', 'Goalkeeper', 232.75)
 ('Tarek', 'Forward', 231.4 ) ('Karim', 'Defender', 220.3 )
 ('Reem', 'Goalkeeper', 219.85) ('Malak', 'Defender', 217.85)]
[('Adam', 'Forward', 252.05) ('Mostafa', 'Defender', 237.  )
 ('Yara', 'Midfielder', 249.5 ) ('Hana', 'Forward', 240.2 )
 ('Ziad', 'Goalkeeper', 232.75) ('Malak', 'Defender', 217.85)
 ('Omar', 'Midfielder', 244.15) ('Farida', 'Forward', 237.  )
 ('Karim', 'Defender', 220.3 ) ('Nada', 'Midfielder', 245.95)
 ('Tarek', 'Forward', 231.4 ) ('Reem', 'Goalkeeper', 219.85)]
['Adam' 'Mostafa' 'Yara' 'Hana' 'Ziad' 'Malak' 'Omar' 'Farida' 'Karim'
 'Nada' 'Tarek' 'Reem']
[252.05 237.   249.5  240.2  232.75 217.85 244.15 237.   220.3  245.95
 231.4  219.85]
[('Adam', 'Forwa

## 6. Select the Squad

Apply this fixed selection rule, exactly as written, so every scout arrives at the same squad:

1. Take the **6 players with the highest `score`**.
2. **Check the rule:** the squad must include **at least 1 Goalkeeper**.
3. **If it doesn't:** remove the *lowest-scoring* player currently in the top 6, and replace them with the *highest-scoring* Goalkeeper (even if that Goalkeeper's score is outside the top 6). **If it already does, leave the squad as is** — don't swap anyone.
4. Print your final 6-player squad (name, position, score).
5. Sanity check: print the average `score` of your final squad vs. the average `score` of the players left out.


In [92]:
# Select your final squad and sanity-check it here
team=sort_score[:6]
print(team)

print(team['position']=="Goalkeeper") ##Not Exist

team = np.delete(team, -1)

goalkeepers = squad[squad["position"] == "Goalkeeper"]
print(goalkeepers)

goalkeeper = sort_score[sort_score["position"] == "Goalkeeper"][0]
print(goalkeeper)

team = np.append(team, goalkeeper)


print(team)

mean_top6=team["score"].mean()

Other=[]

for player in sort_score:
      if player not in team:
        Other.append(player)
        
Other=np.array(Other, dtype=squad.dtype)

print(Other)  
      
Mean_other=Other["score"].mean()  


print(f"Mean Top6 = {mean_top6}, Mean Other = {Mean_other}")        

[('Adam', 'Forward', 252.05) ('Yara', 'Midfielder', 249.5 )
 ('Nada', 'Midfielder', 245.95) ('Omar', 'Midfielder', 244.15)
 ('Hana', 'Forward', 240.2 ) ('Mostafa', 'Defender', 237.  )]
[False False False False False False]
[('Ziad', 'Goalkeeper', 232.75) ('Reem', 'Goalkeeper', 219.85)]
('Ziad', 'Goalkeeper', 232.75)
[('Adam', 'Forward', 252.05) ('Yara', 'Midfielder', 249.5 )
 ('Nada', 'Midfielder', 245.95) ('Omar', 'Midfielder', 244.15)
 ('Hana', 'Forward', 240.2 ) ('Ziad', 'Goalkeeper', 232.75)]
[('Mostafa', 'Defender', 237.  ) ('Farida', 'Forward', 237.  )
 ('Tarek', 'Forward', 231.4 ) ('Karim', 'Defender', 220.3 )
 ('Reem', 'Goalkeeper', 219.85) ('Malak', 'Defender', 217.85)]
Mean Top6 = 244.09999084472656, Mean Other = 227.23333740234375


## 7. Final Report

In 4–6 sentences, write a short summary as if you were sending it to the coach. Include:
- The final squad (names)
- One sentence on why Endurance was weighted highest (in your own words, based on what a tournament format demands)
- Which player was swapped in/out to satisfy the Goalkeeper rule (if applicable), and what that costs the squad in terms of average score

*(Write this as a markdown cell or as comments — either is fine.)*


In [93]:
# Your final report here

"""
1-The final squad (names): ['Adam' 'Yara' 'Nada' 'Omar' 'Hana' 'Ziad']

2-Endurance was given the highest weight because players need to maintain a high level of performance throughout the entire tournament,
especially when playing multiple matches in a short period


3-The original top six did not include a Goalkeeper, so Ziad was added to the squad and Mostafa was removed to satisfy the team selection rule. 
This substitution slightly reduced the squad's average score, 
but it ensured that the team had a complete and valid lineup with a Goalkeeper.
"""


print(team['name'])

['Adam' 'Yara' 'Nada' 'Omar' 'Hana' 'Ziad']
